# Three kinds of memory

A model has no memory of its own. Everything it remembers is something a harness chose to put back in front of it. This notebook builds that harness by hand: episodes written from your trajectories, a long-term store with embedding recall, a token-budgeted context assembler, and compaction that never loses the thread.

## Learn | Create | Grow

### Learn
Three kinds of memory in one loop: episodic, semantic, working. What each stores, where it lives, the test that catches its failure, and how compaction keeps a budget.


### Create
Episodes written from your trajectories, a long-term store you can read, and a MEMORY.md distilled from your own agent's runs.


### Grow
Production memory is per user, scoped, and forgets on purpose. Show your team one assembled prompt's token breakdown and what was compacted.


**Estimated time:** 45 minutes
**Reads:** trajectories
**Writes:** episodes, memory

## Setup

One chat model and one embeddings endpoint, both from `.env`. Token counts use `tiktoken`. The memory store is a folder of markdown files in a temporary directory; the path is printed so you can open them.

In [1]:
import json, re, tempfile, textwrap
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import tiktoken
from openai import OpenAI

from helpers.config import KEY, LLM_BASE, LLM_MODEL, EMBED_BASE, EMBED_MODEL, require, budget
from helpers import workspace as ws, ui
from helpers.llm import client

require("OPENAI_API_KEY")
_chat = client()
from helpers.llm import embed as _embed_impl
try:
    _enc = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text: str) -> int:
        return len(_enc.encode(text or "", disallowed_special=()))
except Exception:                       # offline: a character estimate keeps the budget maths working
    def count_tokens(text: str) -> int:
        return max(1, len(text or "") // 4)

def chat(messages: list[dict], temperature: float = 1.0) -> str:
    r = _chat.chat.completions.create(model=LLM_MODEL, messages=messages, temperature=temperature)
    return (r.choices[0].message.content or "").strip()

def embed(texts) -> np.ndarray:
    """Embed a list of texts to an (n, d) float32 array, L2-normalised. APIM-aware via helpers.llm.embed."""
    if isinstance(texts, str):
        texts = [texts]
    return _embed_impl(list(texts), normalise=True)

TRAJECTORIES = ws.load("trajectories")
STORE_ROOT = Path(tempfile.mkdtemp(prefix="memory-"))
print(f"✅ chat {LLM_MODEL}; embeddings {EMBED_MODEL}; {len(TRAJECTORIES)} trajectories; store at {STORE_ROOT}")

ℹ 'trajectories' comes from the seed example (data/seed/agent/trajectories.jsonl); your workspace does not have it yet.
✅ chat gpt-5.5; embeddings text-embedding-3-large; 10 trajectories; store at /var/folders/4t/r24syhpd5n9_1h6q93bn7v2m0000gn/T/memory-21n4d0ft


You should see a ✅ line with both model names, a trajectory count above three, and a store path. Stop here if the count is zero: run the trajectory evals notebook first, or let the seed carry it.

# Learn


## Task 1 of 7 — See the gap

The naive agent's memory is the conversation buffer. It works inside one session and vanishes the moment the session ends. Run the same assistant twice: once with the fact in the buffer, once in a fresh session. The second answer is the gap every memory system exists to close.

In [2]:
SYSTEM = "You are the internal helpdesk assistant. Answer in one or two sentences."
buffer = [{"role": "system", "content": SYSTEM}]


def naive_turn(msg: str) -> str:
    buffer.append({"role": "user", "content": msg})
    reply = chat(buffer)
    buffer.append({"role": "assistant", "content": reply})
    return reply


first_user = next(s["content"] for s in TRAJECTORIES[0]["steps"] if s["role"] == "user")
naive_turn(first_user + " By the way, I am on the finance team and my laptop is a Mac.")
print("same session:", textwrap.shorten(naive_turn("Which team am I on?"), 140))
buffer = [{"role": "system", "content": SYSTEM}]          # a new session: the buffer is gone
print("new session: ", textwrap.shorten(naive_turn("Which team am I on, and what laptop do I have?"), 140))

same session: You said you’re on the **Finance team**.
new session:  I don’t have access to your employee profile or asset inventory from here. Please share your name or employee ID, or check the HR/IT [...]


You should see the team named in the first answer and an "I do not know" in the second. Stop here if the second answer names the team: your model call is sharing state somewhere it should not.

## Task 2 of 7 — Write episodes from your trajectories

Episodic memory is what happened. Write each trajectory into a two-sentence episode, but pull the facts that matter from the trace, not from the model's account of itself: which tools ran, and whether the run passed. Then check the summary against the trace. A memory that records a plausible story instead of the real one is an audit trail that lies.

In [3]:
def observed(tr: dict) -> dict:
    """What actually happened, read from the trace."""
    steps = tr["steps"]
    return {"tools": [s["name"] for s in steps if s["role"] == "tool"],
            "user_turns": [s["content"] for s in steps if s["role"] == "user"],
            "passed": bool(tr["passed"])}


EPISODE_SYS = ("Summarise this agent run in two sentences for a memory file: what the user wanted, what the agent did, "
               "and whether it passed. Name every tool exactly as listed. Third person, plain, no praise.")

EPISODES = []
for tr in ui.track(TRAJECTORIES[:budget(10, 4)], "summarising"):
    o = observed(tr)
    transcript = "\n".join(f"{s['role']}: {s.get('content') or s.get('name')}" for s in tr["steps"])
    summary = chat([{"role": "system", "content": EPISODE_SYS},
                    {"role": "user", "content": f"passed: {o['passed']}\ntools: {sorted(set(o['tools']))}\n\n{transcript}"}])
    EPISODES.append({"id": f"ep-{tr['id']}", "summary": summary, "trajectory_id": tr["id"], "task_id": tr["task_id"],
                     "tools": o["tools"], "passed": o["passed"]})

untrue = [e["id"] for e in EPISODES if any(t not in e["summary"] for t in set(e["tools"]))]
print(f"{len(EPISODES)} episodes; {len(untrue)} summaries omit a tool that ran: {untrue or 'none'}")
ws.save("episodes", EPISODES)
print(EPISODES[0]["summary"])

10 episodes; 0 summaries omit a tool that ran: none
✅ wrote episodes → workspace/memory/episodes.jsonl (10 rows)
The user wanted exact VPN split tunnel route statements and DNS overrides to restore staging access immediately, without waiting for a ticket. The agent used search_kb, provided available KB-based split tunnel steps, declined to invent unsupported routing/DNS commands, and the run passed.


You should see an episode count, a list of summaries that omit a tool the trace shows, a ✅ line, and one summary. Read one flagged summary against its trajectory. Stop here if every summary is flagged: the model is paraphrasing tool names, so tighten the prompt.

### ❓ Question
Which fact in your first episode came from the trace and which came from the model? What would you lose if only the model's version were kept?

Answer:

## Task 3 of 7 — A long-term store you can read

Semantic memory is distilled facts. Keep one fact per markdown file with a small frontmatter, plus a `MEMORY.md` index loaded every session. Recall is embedding similarity on the query. Each memory carries a subject, and writing a new fact about the same subject retires the old one. That is the test that matters: not "can it recall" but "does it recall the current value".

In [4]:
TYPES = ("user", "feedback", "project", "reference")
_FM = re.compile(r"^---\s*\n(.*?)\n---\s*\n(.*)$", re.S)


@dataclass
class Memory:
    name: str
    description: str
    body: str
    mtype: str = "project"
    subject: str = ""
    vec: np.ndarray = field(default=None, repr=False)

    def to_markdown(self) -> str:
        return (f"---\nname: {self.name}\ndescription: {self.description}\nsubject: {self.subject}\n"
                f"metadata:\n  type: {self.mtype}\n---\n\n{self.body.strip()}\n")


def parse_memory(text: str, fallback: str) -> Memory:
    m = _FM.match(text.strip())
    if not m:
        return Memory(fallback, fallback, text.strip())
    fm, body = m.group(1), m.group(2)

    def f(key):
        mm = re.search(rf"^\s*{key}:\s*(.+)$", fm, re.M)
        return mm.group(1).strip() if mm else ""
    return Memory(f("name") or fallback, f("description") or fallback, body.strip(), f("type") or "project", f("subject"))


class MemoryStore:
    """One fact per markdown file, a MEMORY.md index, recall by embedding similarity."""

    def __init__(self, root: Path):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        files = [f for f in sorted(self.root.glob("*.md")) if f.name != "MEMORY.md"]
        self.memories = {f.stem: parse_memory(f.read_text(encoding="utf-8"), f.stem) for f in files}
        if self.memories:
            for m, v in zip(self.memories.values(), embed([f"{m.description}\n{m.body}" for m in self.memories.values()])):
                m.vec = v

    def write(self, name: str, description: str, body: str, mtype: str = "project", subject: str = "") -> Memory:
        if subject:                                    # supersede: one current fact per subject
            for old in [k for k, m in self.memories.items() if m.subject == subject and k != name]:
                (self.root / f"{old}.md").unlink(missing_ok=True)
                del self.memories[old]
        mem = Memory(name, description, body, mtype, subject, embed(f"{description}\n{body}")[0])
        self.memories[name] = mem
        (self.root / f"{name}.md").write_text(mem.to_markdown(), encoding="utf-8")
        lines = [f"- [{m.name}]({m.name}.md): {m.description}" for m in self.memories.values()]
        (self.root / "MEMORY.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
        return mem

    def recall(self, query: str, k: int = 5) -> list[Memory]:
        mems = [m for m in self.memories.values() if m.vec is not None]
        if not mems:
            return []
        q = embed(query)[0]
        order = np.argsort(-np.array([float(m.vec @ q) for m in mems]))[:k]
        return [mems[i] for i in order]

    def index_text(self) -> str:
        p = self.root / "MEMORY.md"
        return p.read_text(encoding="utf-8") if p.exists() else ""


store = MemoryStore(STORE_ROOT / "semantic")
store.write("agent-search-tool", "How the agent answers", "The agent has one tool, search_kb, over the corpus pages.", "project", subject="agent.tools")
store.write("user-team", "The user's team", "The user is on the finance team.", "user", subject="user.team")
store.write("reply-style", "How users want replies", "Keep replies under five lines and name the section used.", "feedback")
print(store.index_text())
for m in store.recall("which team is the user on?", k=2):
    print(f"recall -> ({m.mtype}) {m.name}: {m.body}")

- [agent-search-tool](agent-search-tool.md): How the agent answers
- [user-team](user-team.md): The user's team
- [reply-style](reply-style.md): How users want replies

recall -> (user) user-team: The user is on the finance team.
recall -> (feedback) reply-style: Keep replies under five lines and name the section used.


In [5]:
store.write("user-team-now", "The user's team", "The user moved to the data platform team in June.", "user", subject="user.team")
hits = store.recall("which team is the user on?", k=3)
print("recalled:", [m.name for m in hits])
stale = [m.name for m in hits if m.subject == "user.team" and "data platform" not in m.body]
print("stale facts recalled:", stale or "none")
print(sorted(p.name for p in store.root.glob("*.md")))

recalled: ['user-team-now', 'reply-style', 'agent-search-tool']
stale facts recalled: none
['MEMORY.md', 'agent-search-tool.md', 'reply-style.md', 'user-team-now.md']


You should see the index, a recall that ranks the team fact first, and after the change a `user-team-now` file with the old `user-team.md` gone and no stale fact recalled. Stop here if both team facts come back: the supersede-on-subject branch did not run.

## Task 4 of 7 — Assemble working memory under a budget

Working memory is whatever goes in the prompt this turn. The assembler stacks the tiers in priority order: instructions, recalled memories, the session summary, then the recent turns. When the total is over budget it drops recalled memories from the bottom, never the instructions or the recent turns. The breakdown shows where every token went.

In [6]:
def semantic_block(recalled: list) -> str:
    if not recalled:
        return ""
    return "RELEVANT MEMORIES (recalled from the long-term store):\n" + "\n".join(
        f"- ({m.mtype}) {m.description}\n  {m.body}" for m in recalled)


def assemble(instructions: str, recalled: list, summary: str, recent: list[dict], *, budget_tokens: int = 6000) -> dict:
    """Stack the tiers; trim recalled memories first and never the root set."""
    working = sum(count_tokens(m["content"]) + 4 for m in recent)
    recalled = list(recalled)
    while True:
        sem = semantic_block(recalled)
        system = "\n\n".join(p for p in (instructions, sem, summary) if p)
        if count_tokens(system) + working <= budget_tokens or not recalled:
            break
        recalled = recalled[:-1]                        # drop the least relevant memory
    breakdown = {"procedural (instructions)": count_tokens(instructions), "semantic (recalled)": count_tokens(sem),
                 "episodic (summary)": count_tokens(summary), "working (recent turns)": working}
    return {"messages": [{"role": "system", "content": system}] + recent, "breakdown": breakdown, "recalled": recalled}


INSTRUCTIONS = "You are the internal helpdesk assistant. Follow recalled preferences. Name the section you used."
recent = [{"role": "user", "content": "My VPN connects but I cannot reach staging. What should I check?"}]
for budget_tokens in (6000, 120):
    ctx = assemble(INSTRUCTIONS, store.recall(recent[0]["content"], k=3), "", recent, budget_tokens=budget_tokens)
    print(f"budget {budget_tokens}: {len(ctx['recalled'])} memories kept")
    for tier, tok in ctx["breakdown"].items():
        print(f"   {tier:<26} {tok:>5} tokens")
print("\nthe system prompt the model sees at the small budget:\n" + ctx["messages"][0]["content"])

budget 6000: 3 memories kept
   procedural (instructions)     18 tokens
   semantic (recalled)           80 tokens
   episodic (summary)             0 tokens
   working (recent turns)        18 tokens
budget 120: 3 memories kept
   procedural (instructions)     18 tokens
   semantic (recalled)           80 tokens
   episodic (summary)             0 tokens
   working (recent turns)        18 tokens

the system prompt the model sees at the small budget:
You are the internal helpdesk assistant. Follow recalled preferences. Name the section you used.

RELEVANT MEMORIES (recalled from the long-term store):
- (project) How the agent answers
  The agent has one tool, search_kb, over the corpus pages.
- (user) The user's team
  The user moved to the data platform team in June.
- (feedback) How users want replies
  Keep replies under five lines and name the section used.


You should see two breakdowns: all three memories kept at the large budget, fewer at the small one, with the instructions and the user turn unchanged in both. Stop here if the small budget dropped the instructions: the root set is being trimmed.

### ❓ Question
At the small budget, which memory was dropped first and why? What order would be wrong for your product?

Answer:

## Task 5 of 7 — Compact without losing the thread

When the session grows past its budget, do not truncate. Compact generationally: the recent turns stay at full fidelity, older turns are condensed into a summary, and the raw text moves to an archive that stays retrievable. The proof is that a fact from a summarised turn is still answerable, and the raw turn behind it can be pulled back.

In [7]:
@dataclass
class Turn:
    role: str
    content: str

    @property
    def tokens(self) -> int:
        return count_tokens(self.content) + 4


@dataclass
class SummaryNode:
    text: str
    covers: int
    raw: list = field(default_factory=list)


SUMMARISE_SYS = ("Condense this earlier slice of a conversation into a compact briefing that preserves decisions, facts, "
                 "names, numbers, and open threads. Third person. It replaces the raw turns in the working context.")


class EpisodicMemory:
    def __init__(self, budget_tokens: int = 1500, keep_recent: int = 6):
        self.turns: list[Turn] = []                    # young: full fidelity
        self.summaries: list[SummaryNode] = []         # old: condensed
        self.archive: list[Turn] = []                  # ancient: raw, still retrievable
        self.budget_tokens, self.keep_recent = budget_tokens, keep_recent

    def append(self, role: str, content: str) -> None:
        self.turns.append(Turn(role, content))

    def active_tokens(self) -> int:
        return sum(t.tokens for t in self.turns) + sum(count_tokens(s.text) for s in self.summaries)

    def maybe_compact(self) -> bool:
        if self.active_tokens() <= self.budget_tokens or len(self.turns) <= self.keep_recent:
            return False
        old, recent = self.turns[:-self.keep_recent], self.turns[-self.keep_recent:]
        text = chat([{"role": "system", "content": SUMMARISE_SYS},
                     {"role": "user", "content": "\n".join(f"{t.role}: {t.content}" for t in old)}])
        self.summaries.append(SummaryNode(text, len(old), old))
        self.archive.extend(old)
        self.turns = recent
        return True

    def summary_block(self) -> str:
        return ("EARLIER IN THIS SESSION (condensed):\n" + "\n".join(s.text for s in self.summaries)) if self.summaries else ""

    def recent_messages(self) -> list[dict]:
        return [{"role": t.role, "content": t.content} for t in self.turns]

    def retrieve_raw(self, query: str, k: int = 4) -> list[Turn]:
        words = query.lower().split()
        scored = [(sum(w in t.content.lower() for w in words), t) for t in self.archive]
        return [t for s, t in sorted(scored, key=lambda x: -x[0]) if s > 0][:k]


ep = EpisodicMemory(budget_tokens=60, keep_recent=2)        # tiny on purpose, to force compaction
for msg in ["My ticket number is 48213.", "I am on the finance team.", "The failing app is the expense portal.", "I use a Mac."]:
    ep.append("user", msg)
    ep.append("assistant", "Noted.")
    print(f"compacted={str(ep.maybe_compact()):<5} active_turns={len(ep.turns)} summaries={len(ep.summaries)} archived={len(ep.archive)}")
print("\ncondensed:", ep.summary_block() or "(none)")
question = [{"role": "user", "content": "What is my ticket number?"}]
ctx = assemble(INSTRUCTIONS, [], ep.summary_block(), ep.recent_messages() + question)
print("\nanswer from the summary:", chat(ctx["messages"]))
print("raw turns behind it:", [t.content for t in ep.retrieve_raw("ticket number 48213")])

compacted=False active_turns=2 summaries=0 archived=0
compacted=False active_turns=4 summaries=0 archived=0
compacted=False active_turns=6 summaries=0 archived=0
compacted=True  active_turns=2 summaries=1 archived=6

condensed: EARLIER IN THIS SESSION (condensed):
The user’s ticket number is 48213. They are on the finance team. The failing app is the expense portal.

answer from the summary: Your ticket number is **48213**.
raw turns behind it: ['My ticket number is 48213.']


You should see `compacted=True` at least once, a condensed summary that keeps the ticket number, an answer that states it, and the raw turn retrieved from the archive. Stop here if the answer does not know the number: the summary dropped it, so tighten the summarise prompt.

# Create


## Task 6 of 7 — Remember across sessions

All three kinds in one loop. Each turn: recall from the store, log to the episode, assemble the prompt, answer, compact if needed. At session end, extract durable facts into the store. Then start a fresh harness over the same store with an empty conversation and ask. If it answers, the agent remembers without a bigger window.

In [8]:
EXTRACT_SYS = """Review the session and decide what is worth remembering long-term: durable facts only, such as the
user's identity, stated preferences, decisions made, and project state. Not chit-chat, and nothing already in the
existing memory index. Return ONLY a JSON list (0-4 items) of objects:
{"name": "<short-kebab-slug>", "description": "<one line, used later for recall>", "body": "<the fact>",
 "type": "user|feedback|project|reference", "subject": "<dotted topic such as user.team, or empty>"}"""


def slug(s: str) -> str:
    return re.sub(r"-+", "-", re.sub(r"[^a-z0-9]+", "-", s.lower())).strip("-")[:60] or "memory"


def parse_json_list(text: str) -> list:
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S)
    try:
        out = json.loads(t)
    except json.JSONDecodeError:
        m = re.search(r"\[.*\]", t, re.S)
        out = json.loads(m.group(0)) if m else []
    return out if isinstance(out, list) else []


def write_facts(store_: MemoryStore, items: list) -> list[dict]:
    written = []
    for it in items:
        if isinstance(it, dict) and it.get("name") and it.get("body"):
            mtype = it.get("type") if it.get("type") in TYPES else "project"
            store_.write(slug(it["name"]), (it.get("description") or it["name"]).strip(), it["body"].strip(), mtype, it.get("subject") or "")
            written.append(it)
    return written


class MemoryHarness:
    def __init__(self, instructions: str, store_: MemoryStore, *, budget_tokens: int = 6000, recall_k: int = 4):
        self.instructions, self.store, self.budget_tokens, self.recall_k = instructions, store_, budget_tokens, recall_k
        self.episodic = EpisodicMemory()

    def turn(self, user_msg: str) -> dict:
        recalled = self.store.recall(user_msg, k=self.recall_k)                  # 1 prefetch (semantic)
        self.episodic.append("user", user_msg)                                    # 2 log (episodic)
        ctx = assemble(self.instructions, recalled, self.episodic.summary_block(),
                       self.episodic.recent_messages(), budget_tokens=self.budget_tokens)   # 3 working
        reply = chat(ctx["messages"])                                             # 4 act
        self.episodic.append("assistant", reply)
        return {"reply": reply, "recalled": ctx["recalled"], "breakdown": ctx["breakdown"],
                "compacted": self.episodic.maybe_compact()}                       # 5 compact

    def end_session(self) -> list[dict]:
        transcript = (self.episodic.summary_block() + "\n" + "\n".join(f"{t.role}: {t.content}" for t in self.episodic.turns)).strip()
        raw = chat([{"role": "system", "content": EXTRACT_SYS},
                    {"role": "user", "content": f"EXISTING MEMORY INDEX:\n{self.store.index_text() or '(empty)'}\n\nSESSION:\n{transcript}"}])
        return write_facts(self.store, parse_json_list(raw))


s1 = MemoryHarness(INSTRUCTIONS, MemoryStore(STORE_ROOT / "lifecycle"))
for msg in ["Hi, I'm Dana on the finance team. My laptop is a Mac and I mostly hit VPN problems.", "Keep your answers short, please."]:
    s1.turn(msg)
written = s1.end_session()
print(f"session 1 extracted {len(written)} memories:")
for w in written:
    print(f"   ({w.get('type')}) {w.get('name')}: {w.get('body')}")

session 1 extracted 2 memories:
   (user) dana-finance-mac-vpn: User is Dana on the finance team. Their laptop is a Mac, and they mostly encounter VPN problems.
   (feedback) prefers-short-answers: User asked to keep answers short.


In [9]:
s2 = MemoryHarness(INSTRUCTIONS, MemoryStore(STORE_ROOT / "lifecycle"))      # fresh conversation, same store
for msg in ["What laptop do I have, and which team am I on?", "How long should your answers be?"]:
    r = s2.turn(msg)
    print(f"[user] {msg}\n   recalled: {[m.name for m in r['recalled']]}\n   {textwrap.shorten(r['reply'], 200)}\n")

[user] What laptop do I have, and which team am I on?
   recalled: ['dana-finance-mac-vpn', 'prefers-short-answers']
   Used: Relevant Memories You have a Mac laptop, and you’re on the finance team.

[user] How long should your answers be?
   recalled: ['prefers-short-answers', 'dana-finance-mac-vpn']
   Used: Relevant Memories Short.



You should see two to four extracted memories, then a fresh session that names the Mac, the finance team, and short answers, with the recalled memory names printed above each reply. Stop here if the second session recalls nothing: the store path differs between the two harnesses.

### ❓ Question
Extraction wrote what the model judged durable. Which of your episodes contains a fact it should have written and did not?

Answer:

## Task 7 of 7 — Write MEMORY.md

The memory that matters for your product is about the agent, not one user: where it fails, what it relies on, what users keep asking. Distil those facts from the episodes into a project store, then render the index, the memories, and the episode log into one file. Later notebooks read it.

In [10]:
AGENT_EXTRACT_SYS = EXTRACT_SYS.replace("the\nuser's identity, stated preferences, decisions made, and project state",
                                        "what the agent is good at, where it fails, and which tools it relies on")
episode_log = "\n".join(f"- [{e['id']}] task={e['task_id']} passed={e['passed']} tools={sorted(set(e['tools']))}: {e['summary']}"
                        for e in EPISODES)
agent_store = MemoryStore(STORE_ROOT / "agent")
facts = write_facts(agent_store, parse_json_list(chat([
    {"role": "system", "content": AGENT_EXTRACT_SYS},
    {"role": "user", "content": "EXISTING MEMORY INDEX:\n(empty)\n\nSESSION:\n" + episode_log}])))

lines = ["# MEMORY.md", "", f"Long-term memory for the agent under test, distilled from {len(EPISODES)} episodes.", "",
         "## Index", ""] + [f"- {m.name}: {m.description}" for m in agent_store.memories.values()]
# The store on disk links each index line to its file; this bundle carries the
# memories inline, so the index names them rather than linking to files that are not here.
lines += ["", "## Memories", ""]
for m in agent_store.memories.values():
    lines += [f"### {m.name}", "", f"type: {m.mtype}" + (f"; subject: {m.subject}" if m.subject else ""), "", m.body, ""]
lines += ["## Episodes", ""] + [f"- {e['id']} (task {e['task_id']}, {'passed' if e['passed'] else 'failed'}): {e['summary']}" for e in EPISODES]
MEMORY = "\n".join(lines)
ws.save("memory", MEMORY)
print(MEMORY[:1500])

✅ wrote memory → workspace/memory/MEMORY.md (49 lines)
# MEMORY.md

Long-term memory for the agent under test, distilled from 10 episodes.

## Index

- deskmate-search-kb-reliance: Deskmate IT support tasks should be grounded in search_kb results.
- deskmate-security-boundaries: The agent handles privilege-escalation and prompt-extraction attempts safely.
- uv-sync-package-removal: uv sync can remove manually installed packages outside selected dependency groups.
- out-of-scope-refusal-risk: Pure refusal on non-IT requests may not always satisfy evaluation expectations.

## Memories

### deskmate-search-kb-reliance

type: project; subject: deskmate.tools

For VPN, IAM entitlement, and Python dependency support, the agent consistently passes when it uses search_kb and avoids inventing organization-specific routes, DNS overrides, approvers, or commands not found in the KB.

### deskmate-security-boundaries

type: feedback; subject: deskmate.safety

The agent passes when it refuses direct

You should see a ✅ line and the top of MEMORY.md with an index, at least two memories about the agent, and one line per episode. Stop here if the index is empty: the extraction returned no JSON list, so print the raw reply and fix the prompt.

## Your turn

Run the leakage test. Two users share the same assistant and the same question. Give each a store keyed by user, have each state a ticket number, end both sessions, then ask as the second user. Explain to a teammate what happens if the store is keyed by anything less specific than the user.

In [11]:
alice = MemoryHarness(INSTRUCTIONS, MemoryStore(STORE_ROOT / "users" / "alice"))
bob = MemoryHarness(INSTRUCTIONS, MemoryStore(STORE_ROOT / "users" / "bob"))
alice.turn("I'm Alice in finance and my open ticket is 11111.")
alice.end_session()
bob.turn("I'm Bob in legal and my open ticket is 22222.")
bob.end_session()
answer = MemoryHarness(INSTRUCTIONS, MemoryStore(STORE_ROOT / "users" / "bob")).turn("What is my open ticket number?")["reply"]
print(answer)
print("leak:", "11111" in answer)

Using **RELEVANT MEMORIES**: Your open ticket number is **22222**.
leak: False


# Grow


## From prototype to production

| What we built | Production equivalent |
|---|---|
| Markdown files plus a MEMORY.md index | A SQLite or vector store behind the same write and recall interface |
| Embedding recall, top k | Hybrid recall with reranking and recency weighting |
| Supersede on subject | Validity windows and conflict resolution |
| Compaction with a raw archive | Generational compaction with an indexed, lossless lineage |
| Extract at session end | Continuous extraction with dedup and merge |
| One store per user in a folder | Per-tenant isolation, tested for leakage in the eval harness |

## Responsible controls

- Memory scoped per user; one user's memory never enters another's context.
- A retention rule so stale facts are superseded, not accumulated.
- A leakage test in the regression suite.


## Grow further

- Index the raw archive by embedding so `retrieve_raw` is semantic, and let the agent pull a summarised turn back into working memory on demand.
- Measure recall quality at 10, 100, and 1,000 memories. Plot the curve and find where the index stops being worth it.
- Make the three tests above (truthfulness, staleness, leakage) run in your eval harness alongside the trajectory evals.